In [ ]:
import platform

%pip install "torch>=2.1.1" "transfromers>=4.45" modelscope "diffusers>0.31.0"  --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "git+https://github.com/huggingface/optimum-intel.git"  --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -qU "openvino>2024.4.0" "openvino-tokenizers" "openvino-genai"

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

In [ ]:
from ipython import Markdown
from pathlib import Path
from IPython.display import Markdown, display

In [ ]:
text_model_id = "iic/nlp_bert_sentiment-analysis_english-base"
text_model_path = Path(text_model_id.split("/")[-1])
ov_text_model_path = text_model_path / "ov"

download_command = f"modelscope download {text_model_id} --local_dir {text_model_path}"
display(Markdown("**Download command:**"))
display(Markdown(f"`{download_command}`"))

if not text_model_path.exists():
    !{download_command}


In [ ]:
export_command = f"optimum-cli export openvino -m {text_model_path} --task text-classification {ov_text_model_path}"
display(Markdown("**Export command:**"))
display(Markdown(f"`{export_command}`"))

if not ov_text_model_path.exists():
    !export_command

In [ ]:
from notebook_utils import device_widget

text_cls_device = device_widget("CPU", "NPU")

text_cls_device

In [ ]:
from transformers import AutoTokenizer
from optimum.intel.openvino import OVModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(ov_text_model_path)
ov_model = OVModelForSequenceClassification.from_pretrained(ov_text_model_path, text_cls_device.value)

input_text = 'Good night.'
input_data = tokenizer(input_text, return_tensors="pt")

output = ov_model(**input_data)
predicted_label_id = output.logits[0].argmax().item()

predicted_label = ov_model.config.id2label[predicted_label_id]

print(f"predicted label: {predicted_label}")


In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

llm_path = Path("Qwen2.5-1.5B-Instruct")
ov_llm_path = llm_path / "ov"
download_command = f"modelscope download {model_id} --local_dir {llm_path}"
display(Markdown("**Download command:**"))
display(Markdown(f"`{download_command}`"))

if not llm_path.exists():
    !{download_command}

In [ ]:
export_command = f"optimum-cli export openvino -m {llm_path} --task text-generation-with-past --weight-format int4 --trust-remote-code {ov_llm_path}"
display(Markdown("**Export command:**"))
display(Markdown(f"`{export_command}`"))

if not ov_llm_path.exists():
    !export_command

In [ ]:
from notebook_utils import device_widget

llm_device = device_widget("CPU")

llm_device

In [ ]:
import openvino_genai as ov_genai

llm_pipe = ov_genai.LLMPipeline(ov_llm_path, llm_device.value)

print(llm_pipe.generate("The Sun is yellow because", max_new_tokens=30))

In [ ]:
import gc

del llm_pipe
gc.collect();

In [ ]:
vlm_model_id = "OpenGVLab/InternVL2-1B"
vlm_path = Path("InternVL2-1B")

ov_vlm_path = vlm_path / "ov"
download_command = f"modelscope download {vlm_model_id} --local_dir {vlm_path}"
display(Markdown("**Download command:**"))
display(Markdown(f"`{download_command}`"))
if not vlm_path.exists():
    !{download_command}

In [ ]:
export_command = f"optimum-cli export openvino -m {vlm_path} --task image-text-to-text --trust-remote-code {ov_vlm_path}"
display(Markdown("**Export command:**"))
display(Markdown(f"`{export_command}`"))

if not ov_vlm_path.exists():
    !export_command

In [ ]:
vlm_device = device_widget("CPU", ["NPU"])
vlm_device

In [ ]:
import requests
import numpy as np
from PIL import Image
import openvino as ov
from io import BytesIO

def load_image(image_file):
    if image_file.startswith("http") or image_file.startswith("https"):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_file).convert("RGB")
    image_data = np.array(image.getdata()).reshape(1, image.size[1], image.size[0], 3).astype(np.byte)
    return image, ov.Tensor(image_data)


vlm_pipe = ov_genai.VLMPipeline(ov_vlm_path, vlm_device.value)
image, image_tensor = load_image("https://github.com/openvinotoolkit/openvino_notebooks/assets/29454499/d5fbbd1a-d484-415c-88cb-9986625b7b11")
prompt = "Briefly describe image"

display(image)
print(vlm_pipe.generate(prompt, image=image_tensor, max_new_tokens=50))

In [ ]:
del vlm_pipe
gc.collect();